# Detecting AI-Generated News Articles

A text classifier that distinguishes human-written news articles from LLM-generated
ones, built and evaluated on a balanced ~10k-article dataset (real *Guardian* articles
vs. LLM-generated articles in the same style).

The notebook works through four detection strategies and compares them on held-out data:

1. **Zero-shot LLM baseline** — ask an LLM to judge authorship directly *(optional, needs an API key)*
2. **Heuristic features** — stylometric signals (sentence-length variance, quote frequency, type-token ratio, etc.) + logistic regression
3. **Perplexity** — how "surprised" GPT-2 is by the text, added as a feature
4. **TF-IDF (unigrams + bigrams)** — the strongest model, selected as the final submission

It closes with an **adversarial-robustness** test (can a crafted prompt fool the detector?)
and a discussion of what that implies for deploying detection at scale.

> Course project — Harvard Kennedy School, DPI-681 (*Public Problem Solving with Generative AI*).
> The core classifier (heuristics + perplexity + TF-IDF) runs with no API key; only the
> optional LLM-baseline and adversarial sections call an LLM.

## Setup & data

In [ ]:
# Install dependencies
!pip install openai transformers torch scikit-learn pandas numpy tqdm --quiet

import pandas as pd
import numpy as np
import re, requests, warnings, asyncio
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Public dataset: ~10k news articles, half from The Guardian (pre-2022, human),
# half generated by an LLM in the Guardian's style.
DATASET_URL = "https://raw.githubusercontent.com/calisley/dpi-681/refs/heads/main/data/assignment5/train_set.csv"
HOLDOUT_URL = "https://raw.githubusercontent.com/calisley/dpi-681/refs/heads/main/data/assignment5/holdout_set.csv"
print("Libraries loaded.")

In [ ]:
df = pd.read_csv(DATASET_URL)
if "body_text" in df.columns and "text" not in df.columns:
    df = df.rename(columns={"body_text": "text"})
df["label"] = df["label"].map({"ai": "AI", "human": "Human", "AI": "AI", "Human": "Human"})
print(f"Loaded {len(df):,} articles | {df['label'].value_counts().to_dict()}")
df.head(3)

In [ ]:
# Train / validation split (hold out 20% for local evaluation)
train_df, val_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df["label"])
train_df = train_df.reset_index(drop=True); val_df = val_df.reset_index(drop=True)
y_train = (train_df["label"] == "AI").astype(int).values
y_val   = (val_df["label"] == "AI").astype(int).values
print(f"Train: {len(train_df):,} | Validation: {len(val_df):,}")

## Step 0 — Can an LLM detect AI writing? *(optional baseline)*

Before building anything, a sanity check: ask an LLM to label authorship directly.
**Requires `OPENAI_API_KEY`.**

> *Result:* the zero-shot LLM hovered around ~60% and was heavily biased toward predicting
> "human" — motivating a purpose-built classifier.

In [ ]:
# Set up the OpenAI client (used only for the optional LLM-baseline and
# adversarial-robustness sections; the core classifier below needs no API key).
import os
from openai import OpenAI, AsyncOpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
aclient = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print("OpenAI client ready.")

In [ ]:
# Classify a sample of 50 articles using GPT-5-nano
# This is fully scaffolded — just run it

llm_sample = val_df.sample(n=50, random_state=42).reset_index(drop=True)

async def classify_with_llm(text: str) -> str:
    """Ask GPT-5-nano whether this article was written by a human or AI."""
    response = await aclient.chat.completions.create(
        model="gpt-5-nano",
        messages=[{
            "role": "user",
            "content": (
                "Read the following news article and determine whether it was "
                "written by a human journalist or generated by an AI language model.\n\n"
                "Respond with ONLY one word: either 'Human' or 'AI'.\n\n"
                f"Article:\n{text[:3000]}"
            )
        }]
    )
    answer = response.choices[0].message.content.strip().lower()
    if 'human' in answer:
        return 'Human'
    elif 'ai' in answer:
        return 'AI'
    return 'Unknown'

async def classify_all():
    tasks = [classify_with_llm(row['text']) for _, row in llm_sample.iterrows()]
    return await asyncio.gather(*tasks)

llm_predictions = await classify_all()

llm_sample['llm_pred'] = list(llm_predictions)
print("Done!")

In [ ]:
# How did the LLM do?
llm_acc = (llm_sample['llm_pred'] == llm_sample['label']).mean()
print(f"GPT-5-nano Accuracy: {llm_acc:.1%}  ({int(llm_acc * len(llm_sample))}/{len(llm_sample)} correct)")
print(pd.crosstab(llm_sample['label'], llm_sample['llm_pred'], margins=True))

mistakes = llm_sample[llm_sample['llm_pred'] != llm_sample['label']]
print(f"\n{len(mistakes)} mistakes out of {len(llm_sample)} articles")

## Step 1 — Heuristic classifier

The idea: AI-generated text carries subtle stylistic fingerprints. Two features are
pre-built; I added three more based on patterns I noticed reading samples from each class:
**sentence-length variance**, **quote frequency**, and **type-token ratio** (lexical diversity).

In [ ]:
# ── Pre-built features ────────────────────────────────────────────────────

def avg_sentence_length(text: str) -> float:
    """Average number of words per sentence."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    if len(sentences) == 0:
        return 0.0
    return np.mean([len(s.split()) for s in sentences])


def count_em_dashes(text: str) -> int:
    """Count dashes — a punctuation mark AI sometimes over- or under-uses."""
    return text.count('\u2014') + text.count('\u2013') + text.count(' - ')


print("Pre-built feature functions defined.")

In [ ]:
import re
import numpy as np

def sentence_length_variance(text: str) -> float:
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    if len(sentences) <= 1:
        return 0.0
    lengths = [len(s.split()) for s in sentences]
    return float(np.var(lengths))

def quote_frequency(text: str) -> float:
    words = text.split()
    if len(words) == 0:
        return 0.0
    quote_count = text.count('"') + text.count('“') + text.count('”')
    return (quote_count / len(words)) * 100

def type_token_ratio(text: str) -> float:
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    if len(words) == 0:
        return 0.0
    return len(set(words)) / len(words)

In [ ]:
# Build the feature matrix — add your custom features here!

def extract_features(texts: pd.Series) -> pd.DataFrame:
    """Extract all features from a Series of article texts."""
    features = pd.DataFrame()

    # Pre-built features
    features['avg_sentence_len'] = texts.apply(avg_sentence_length)
    features['em_dash_count'] = texts.apply(count_em_dashes)
    features['word_count'] = texts.apply(lambda x: len(x.split()))

    features['sentence_len_var'] = texts.apply(sentence_length_variance)
    features['quote_freq'] = texts.apply(quote_frequency)
    features['type_token_ratio'] = texts.apply(type_token_ratio)

    return features


print("Extracting features...")
X_train_h = extract_features(train_df['text'])
X_val_h = extract_features(val_df['text'])
print(f"Feature matrix: {X_train_h.shape[0]} articles \u00d7 {X_train_h.shape[1]} features")
X_train_h.describe().round(2)

In [ ]:
# Train a logistic regression on the heuristic features

scaler_h = StandardScaler()
X_train_h_s = scaler_h.fit_transform(X_train_h)
X_val_h_s = scaler_h.transform(X_val_h)

clf_heuristic = LogisticRegression(max_iter=1000, random_state=42)
clf_heuristic.fit(X_train_h_s, y_train)

y_pred_h = clf_heuristic.predict(X_val_h_s)
y_prob_h = clf_heuristic.predict_proba(X_val_h_s)[:, 1]

print(f"Heuristic Classifier \u2014 Validation Results")
print(f"  Accuracy: {accuracy_score(y_val, y_pred_h):.1%}")
print(f"  AUC:      {roc_auc_score(y_val, y_prob_h):.4f}")

In [ ]:
# Which features matter most?
print("Feature weights (positive = more likely AI):")
for feat, w in sorted(zip(X_train_h.columns, clf_heuristic.coef_[0]), key=lambda x: -abs(x[1])):
    direction = "\u2192 AI" if w > 0 else "\u2192 Human"
    bar = '\u2588' * int(abs(w) * 3)
    print(f"  {feat:<22s}  {w:+.3f}  {bar}  {direction}")

**Reflection.** The heuristic model reached **85.8% accuracy** vs. the LLM's ~60% —
it's purpose-built to pick up on the patterns that separate human and AI writing. The
feature weights mostly matched intuition: I was initially surprised type-token ratio mattered
most, but it makes sense — LLMs work hard to avoid repetition, producing more uniform vocabulary.
High quote frequency in AI text was the bigger surprise, and it raises a real question about where
those fabricated-sounding quotes actually come from.

## Step 2 — Perplexity

Perplexity measures how predictable a text is to a language model (GPT-2 here). Lower
perplexity ≈ more predictable. I compute it for every article and add it as a feature.

In [ ]:
# Load GPT-2 and set up perplexity computation
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cpu':
    print("💡 Tip: Runtime → Change runtime type → T4 GPU will make this ~5x faster.")

print("Loading GPT-2...")
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
model_gpt2 = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
model_gpt2.eval()
print("Ready.")

In [ ]:
def compute_perplexity(text: str, max_length: int = 1024) -> float:
    """How surprised is GPT-2 by this text? Lower = more predictable."""
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length)
    ids = enc.input_ids.to(device)
    with torch.no_grad():
        loss = model_gpt2(ids, labels=ids).loss
    return torch.exp(loss).item()

# Quick test
test_ppl = compute_perplexity("The president of the United States held a press conference today.")
print(f"Test perplexity: {test_ppl:.1f}")
print("Function defined.")

In [ ]:
from tqdm import tqdm
train_df["perplexity"] = [compute_perplexity(t) for t in tqdm(train_df["text"])]
val_df["perplexity"]   = [compute_perplexity(t) for t in tqdm(val_df["text"])]
train_df.groupby("label")["perplexity"].mean()

In [ ]:
# Perplexity distribution — do the classes separate?

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [('Human', 'steelblue'), ('AI', 'coral')]:
    subset = train_df[train_df['label'] == label]['perplexity'].clip(upper=500)
    ax.hist(subset, bins=50, alpha=0.6, color=color, label=label, density=True)
ax.set_xlabel('Perplexity (lower = more predictable)')
ax.set_ylabel('Density')
ax.set_title('Perplexity: Human vs AI')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nPerplexity by label:")
print(train_df.groupby('label')['perplexity'].describe().round(1))

**Reflection.** Perplexity separates the classes less cleanly than I expected — there's
substantial overlap, partly because GPT-2 is older/weaker. Some human articles have low perplexity
because journalistic writing is concise and standardized. Useful, but not decisive on its own.

## Step 3 — Combined model (heuristics + perplexity)

In [ ]:
# Build the combined feature matrix and train a logistic regression.
# Use the heuristic classifier code from Step 1 as a template!

X_train_combined = X_train_h.copy()
X_val_combined = X_val_h.copy()

X_train_combined['perplexity'] = train_df['perplexity'].values
X_val_combined['perplexity'] = val_df['perplexity'].values

scaler_c = StandardScaler()
X_train_combined_s = scaler_c.fit_transform(X_train_combined)
X_val_combined_s = scaler_c.transform(X_val_combined)

clf_combined = LogisticRegression(max_iter=1000, random_state=42)
clf_combined.fit(X_train_combined_s, y_train)

y_pred_c = clf_combined.predict(X_val_combined_s)
y_prob_c = clf_combined.predict_proba(X_val_combined_s)[:, 1]

In [ ]:
# Combined model evaluation

print(f"Combined Classifier (Heuristics + Perplexity) — Validation Results")
print(f"  Accuracy: {accuracy_score(y_val, y_pred_c):.1%}")
print(f"  AUC:      {roc_auc_score(y_val, y_prob_c):.4f}")
print()
print(classification_report(y_val, y_pred_c, target_names=['Human', 'AI']))

# Compare with heuristic-only model
print(f"\nComparison:")
print(f"  Heuristic-only AUC:     {roc_auc_score(y_val, y_prob_h):.4f}")
print(f"  Combined AUC:           {roc_auc_score(y_val, y_prob_c):.4f}")
improvement = roc_auc_score(y_val, y_prob_c) - roc_auc_score(y_val, y_prob_h)
print(f"  Improvement:            {improvement:+.4f}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm_c = confusion_matrix(y_val, y_pred_c)
ConfusionMatrixDisplay(cm_c, display_labels=['Human', 'AI']).plot(cmap='Blues', ax=ax)
ax.set_title(f'Combined — AUC {roc_auc_score(y_val, y_prob_c):.3f}')
plt.tight_layout()
plt.show()

**Reflection.** Adding perplexity raised AUC from **0.9166 → 0.9391** — a modest but real
gain, showing perplexity carries signal the heuristics miss. The combined model makes more of its
mistakes on *human* articles misclassified as AI, likely because professional journalistic prose
can read as machine-like.

## Step 4 — TF-IDF model (final submission)

I then tried a TF-IDF representation (unigrams + bigrams) with logistic regression. It
**outperformed the heuristic + perplexity model**, so I selected it as my final classifier.

In [ ]:
# Your improved model here — anything goes!

# Step 1: Import
from sklearn.feature_extraction.text import TfidfVectorizer

# Step 2: Vectorize text
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),   # unigrams + bigrams
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_val_tfidf = tfidf.transform(val_df['text'])

# Step 3: Train model
clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = clf_tfidf.predict(X_val_tfidf)
y_prob_tfidf = clf_tfidf.predict_proba(X_val_tfidf)[:, 1]

# Step 4: Evaluate
print("TF-IDF Model — Validation Results")
print(f"  Accuracy: {accuracy_score(y_val, y_pred_tfidf):.1%}")
print(f"  AUC:      {roc_auc_score(y_val, y_prob_tfidf):.4f}")

## Adversarial robustness — can a crafted prompt fool the detector?

A motivated user can adapt prompts to evade detection. Here I generate an article designed
to look human and run it through the classifiers. **Requires `OPENAI_API_KEY`.**

In [ ]:
# Craft a prompt that tries to make an LLM produce text that evades the detector
prompt = ("Write a 400-word news article in the style of a human journalist. "
          "Vary your sentence lengths, include a couple of quotes, and intentionally "
          "introduce small stylistic inconsistencies so it does not read as machine-generated.")

In [ ]:
# Step 2 — Generate the article (read-only output)
# GPT will produce an article based on your prompt. You can look at it but you can't edit it!

from IPython.display import display, HTML

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[{"role": "user", "content": prompt}]
)

generated_article = response.choices[0].message.content.strip()

display(HTML(
    '<div>'
    f'<p style="color:#888; font-size:12px; margin-bottom:8px;">📝 Generated article ({len(generated_article.split())} words)</p>'
    f'{generated_article.replace(chr(10), "<br>")}'
    '</div>'
))
print("\n⬇️  Run the next cell to see if your classifiers are fooled.")

In [ ]:
results = {}

# ── Heuristic classifier ──────────────────────────────────────────────────────────
try:
    feats = extract_features(pd.Series([generated_article]))
    feats_s = scaler_h.transform(feats)
    prob = clf_heuristic.predict_proba(feats_s)[0, 1]
    pred = "AI" if prob >= 0.5 else "Human"
    results['Heuristic'] = (pred, prob)
except Exception:
    results['Heuristic'] = None

# ── Combined classifier (heuristics + perplexity) ────────────────────────────────────
try:
    feats = extract_features(pd.Series([generated_article]))
    ppl = compute_perplexity(generated_article)
    feats['perplexity'] = ppl
    feats_s = scaler_c.transform(feats)
    prob = clf_combined.predict_proba(feats_s)[0, 1]
    pred = "AI" if prob >= 0.5 else "Human"
    results['Combined'] = (pred, prob, ppl)
except Exception:
    results['Combined'] = None

# ── Results ───────────────────────────────────────────────────────────────────────────
print("=" * 55)
print("  CLASSIFIER VERDICTS")
print("=" * 55)

fooled_any = False
for name, res in results.items():
    if res is None:
        print(f"  {name:<14s}  ⚠️  (not available — did you run that step?)")
        continue
    pred = res[0]
    prob = res[1]
    if pred == "Human":
        print(f"  {name:<14s}  🎉 FOOLED!  (P(AI) = {prob:.1%})")
        fooled_any = True
    else:
        print(f"  {name:<14s}  🔍 Caught.  (P(AI) = {prob:.1%})")
    if len(res) > 2:
        print(f"  {'':14s}  Perplexity: {res[2]:.1f}")

print("=" * 55)
if fooled_any:
    print("\n🎉  Nice! You fooled at least one classifier.")
    print("    Try running it again with a different prompt — can you do it consistently?")
else:
    print("\n🔍  Your classifiers caught this one. Try a different prompt strategy!")
    print("    Go back to Step 1 and iterate on your prompt.")

**Reflection.** The most effective evasion was instructing the model to prioritize
human-like *imperfections* over clarity — exploiting the fact that my classifier associates
uniform structure and high lexical uniformity with "AI." That tells me it detects *patterns*,
not AI per se: small prompt changes flip the prediction, which is a fundamental limitation of
AI-text detection as a governance tool.

## Holdout predictions

In [ ]:
# Load the holdout set and score it with the combined model
holdout = pd.read_csv(HOLDOUT_URL)
if "body_text" in holdout.columns and "text" not in holdout.columns:
    holdout = holdout.rename(columns={"body_text": "text"})
print(f"Loaded holdout set: {len(holdout)} articles")

In [ ]:
# Generate predictions for the holdout set using the combined model
holdout_feats = extract_features(holdout["text"])
holdout_feats["perplexity"] = [compute_perplexity(t) for t in tqdm(holdout["text"])]
holdout_feats_s = scaler_c.transform(holdout_feats)
holdout_probs = clf_combined.predict_proba(holdout_feats_s)[:, 1]

holdout_preds = pd.DataFrame({"id": holdout["id"], "predicted_probability": holdout_probs})
holdout_preds.to_csv("holdout_predictions.csv", index=False)
print(holdout_preds.head())

## Summary

| Approach | Validation result |
|---|---|
| Zero-shot LLM | ~60% accuracy (biased toward "human") |
| Heuristic features + LR | 85.8% accuracy, 0.9166 AUC |
| + Perplexity | 0.9391 AUC |
| **TF-IDF (uni+bi) + LR** | **best — selected as final** |

**Takeaway.** A lightweight, interpretable classifier beats a frontier LLM at this narrow task,
but every approach is brittle to adversarial prompting — a key caveat for anyone considering AI-text
detection for content moderation or disclosure enforcement.